In [2]:
import sqlite3
import pandas as pd

# Connect to an in-memory SQLite database
conn = sqlite3.connect(':memory:')
cursor = conn.cursor()

# Load data from the CSV file
try:
    df_events = pd.read_csv('/home/events_sample.csv')
    print("CSV file 'events_sample.csv' loaded successfully.")
except FileNotFoundError:
    print("Error: 'events_sample.csv' not found. Please ensure the file is in the correct path.")
    # Optionally, handle the error or exit if the file is crucial
    # For now, we'll continue with an empty DataFrame if not found
    df_events = pd.DataFrame(columns=['user_id', 'event_type']) # Create an empty DataFrame with expected columns

# Create the 'events' table from the DataFrame's schema and insert data
df_events.to_sql('events', conn, if_exists='replace', index=False)

print("Data from 'events_sample.csv' inserted into 'events' table.")

CSV file 'events_sample.csv' loaded successfully.
Data from 'events_sample.csv' inserted into 'events' table.


In [3]:
# Now, execute the SQL funnel query
sql_query = """
WITH user_funnel AS (
    SELECT
        user_id,
        MAX(CASE WHEN event_type = 'view' THEN 1 ELSE 0 END) AS viewed,
        MAX(CASE WHEN event_type = 'add_to_cart' THEN 1 ELSE 0 END) AS added_to_cart,
        MAX(CASE WHEN event_type = 'purchase' THEN 1 ELSE 0 END) AS purchased
    FROM events
    GROUP BY user_id
)

SELECT
    COUNT(*) AS total_users,
    SUM(viewed) AS viewed_users,
    SUM(added_to_cart) AS cart_users,
    SUM(purchased) AS purchased_users,

    ROUND(SUM(added_to_cart)*1.0 / SUM(viewed), 3) AS view_to_cart_rate,
    ROUND(SUM(purchased)*1.0 / SUM(added_to_cart), 3) AS cart_to_purchase_rate

FROM user_funnel;
"""

# Use pandas to read the SQL query result directly into a DataFrame
df_funnel = pd.read_sql_query(sql_query, conn)
display(df_funnel)

# Close the connection
conn.close()

,total_users,viewed_users,cart_users,purchased_users,view_to_cart_rate,cart_to_purchase_rate
0,500,500,416,268,0.832,0.644
